In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 1 — Mount Drive & Set Path
### ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
print(f"Python Version: {sys.version.split()[0]}")
print(f"Running inside Google Colab: {IN_COLAB}")

# Clone repository if running inside Colab
if IN_COLAB:
    if not os.path.exists("thesis-on-keystroke-dynamics-authentication"):
        !git clone https://github.com/itsrashedsatter/thesis-on-keystroke-dynamics-authentication.git
        %cd thesis-on-keystroke-dynamics-authentication
    else:
        %cd thesis-on-keystroke-dynamics-authentication
        !git pull origin main

# Install required dependencies
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy joblib pyarrow tensorflow

# Verify GPU acceleration
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow Version: {tf.__version__}")
if gpus:
    print(f"✅ GPU Acceleration ACTIVE: {gpus}")
else:
    print("ℹ️ Running on CPU. (Tip: Colab Menu -> Runtime -> Change runtime type -> Select 'T4 GPU' for faster training)")


# ⌨️ Keystroke Dynamics Continuous Authentication Pipeline
## Scaled 1,000-User Cohort Benchmark & Zero-Trust Telemetry Architecture

This interactive notebook executes the complete end-to-end continuous biometric authentication pipeline:
- **Cohort Scale**: 1,000 Typists $\times$ 15 Sessions = 15,000 Total Typing Sessions (730,950 Keystrokes).
- **Stage 01 — Merged Telemetry**: Verifies raw keystroke events, user metadata, and session integrity.
- **Stage 02 — Data Cleaning & Outlier Removal**: Filters corrupted timings, computes $HT, UD, DD$, flags rollovers ($UD < 0$) and cognitive pauses ($DD > 5{,}000$ ms).
- **Stage 03 — Feature Engineering**: 124 biometric digraph features per session, zero-leakage stratified 80/20 split (12,000 Train / 3,000 Test).
- **Stage 04 — Model Training**: Benchmarks **Random Forest**, **1D-CNN**, and **Bidirectional LSTM** across $C = 1,000$ classes.
- **Evaluation & Zero-Trust Sweeps**: Full confusion matrices, One-vs-Rest Macro ROC/AUC, precision/recall/F1 metrics, Top 20 Gini feature importances, and continuous authentication FAR/FRR/EER threshold sweeps.


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 2 — Install and Import Libraries & Styling Setup
### ─────────────────────────────────────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib, json, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved → {path}')
    plt.show()

# Base path auto-detection: checks if cohort_1000/ exists, or if running directly inside cohort_1000/
BASE_DIR = 'cohort_1000' if os.path.exists('cohort_1000') else '.'
print(f"Data Root Directory set to: '{BASE_DIR}'")


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 3 — STAGE 01: Merged Dataset Verification (1,000 Users)
### ─────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

# Load the merged raw keystroke dataset (supports fast Parquet or CSV)
raw_parquet = os.path.join(BASE_DIR, "01_data_merge/merged_raw_keystrokes.parquet")
raw_csv = os.path.join(BASE_DIR, "01_data_merge/merged_raw_keystrokes.csv")

if os.path.exists(raw_parquet):
    print(f"⚡ Loading optimized raw telemetry from {raw_parquet}...")
    df_raw = pd.read_parquet(raw_parquet)
else:
    print(f"Loading raw telemetry from {raw_csv}...")
    df_raw = pd.read_csv(raw_csv, encoding='latin-1')

print("==================================================")
print("📊 STAGE 01: MERGED DATASET VERIFICATION (1,000-USER COHORT):")
print(f"   - Total Raw Keystrokes Merged: {len(df_raw):,}")
print(f"   - Experimental Cohort Typists: {df_raw['PARTICIPANT_ID'].nunique():,} participants")
print(f"   - Sessions Per Typist:         {df_raw.groupby('PARTICIPANT_ID')['TEST_SECTION_ID'].nunique().iloc[0]} sessions each")
print(f"   - Total Unique Sessions:       {df_raw.groupby(['PARTICIPANT_ID', 'TEST_SECTION_ID']).ngroups:,}")
print(f"   - Merged Columns ({len(df_raw.columns)}):      {list(df_raw.columns[:6])} ... + metadata")
print("==================================================")

# Display sample telemetry and merged metadata
cols_to_show = [c for c in ['PARTICIPANT_ID', 'TEST_SECTION_ID', 'KEYSTROKE_ID', 'PRESS_TIME', 'RELEASE_TIME', 'LETTER', 'KEYCODE', 'AGE', 'GENDER', 'AVG_WPM_15'] if c in df_raw.columns]
display(df_raw[cols_to_show].head(5))


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 4 — STAGE 02: Telemetry Cleaning & Interval Calculation
### ─────────────────────────────────────────────────────────────────────
# 1. Handle Missing Values
initial_rows = len(df_raw)
df_clean = df_raw.copy()

# Space character imputation
if 'LETTER' in df_clean.columns:
    space_mask = df_clean['LETTER'].isnull() & (df_clean['KEYCODE'] == 32)
    df_clean.loc[space_mask, 'LETTER'] = ' '
    df_clean['LETTER'] = df_clean['LETTER'].fillna('UNKNOWN')

# 2. Remove Duplicate Keystroke Events
dup_mask = df_clean.duplicated(subset=['PARTICIPANT_ID', 'TEST_SECTION_ID', 'KEYSTROKE_ID'], keep='first')
df_clean = df_clean[~dup_mask].copy()

# 3. Chronological Sorting
df_clean = df_clean.sort_values(by=['PARTICIPANT_ID', 'TEST_SECTION_ID', 'PRESS_TIME', 'KEYSTROKE_ID']).reset_index(drop=True)

# 4. Calculate Timing Intervals
df_clean['HOLD_TIME'] = df_clean['RELEASE_TIME'] - df_clean['PRESS_TIME']
df_clean['PREV_PRESS_TIME'] = df_clean.groupby(['PARTICIPANT_ID', 'TEST_SECTION_ID'])['PRESS_TIME'].shift(1)
df_clean['PREV_RELEASE_TIME'] = df_clean.groupby(['PARTICIPANT_ID', 'TEST_SECTION_ID'])['RELEASE_TIME'].shift(1)

df_clean['UD_TIME'] = df_clean['PRESS_TIME'] - df_clean['PREV_RELEASE_TIME']
df_clean['DD_TIME'] = df_clean['PRESS_TIME'] - df_clean['PREV_PRESS_TIME']
df_clean['IS_FIRST_KEY'] = df_clean['PREV_PRESS_TIME'].isnull().astype(int)
df_clean.drop(columns=['PREV_PRESS_TIME', 'PREV_RELEASE_TIME'], inplace=True)

# 5. Physical Validity Filters
invalid_hold = df_clean['HOLD_TIME'] <= 0
corrupted_hold = df_clean['HOLD_TIME'] > 3000
df_clean = df_clean[~(invalid_hold | corrupted_hold)].copy().reset_index(drop=True)

# 6. Flag Key Rollovers (UD < 0) and Cognitive Pauses (DD > 5000 ms)
df_clean['IS_ROLLOVER'] = ((df_clean['UD_TIME'] < 0) & (df_clean['IS_FIRST_KEY'] == 0)).astype(int)
df_clean['IS_LONG_PAUSE'] = ((df_clean['DD_TIME'] > 5000) & (df_clean['IS_FIRST_KEY'] == 0)).astype(int)

final_rows = len(df_clean)
retention = (final_rows / initial_rows) * 100

print("==================================================")
print("✅ STAGE 02 DATA CLEANING COMPLETE (1,000 COHORT):")
print(f"   - Initial Rows:        {initial_rows:,}")
print(f"   - Corrupted Dropped:   {(initial_rows - final_rows):,} rows ({((initial_rows-final_rows)/initial_rows)*100:.3f}%)")
print(f"   - Final Cleaned Rows:  {final_rows:,}")
print(f"   - Retention Rate:      {retention:.2f}%")
print(f"   - Rollover Events:     {df_clean['IS_ROLLOVER'].sum():,} flagged ({df_clean['IS_ROLLOVER'].mean()*100:.2f}% of transitions)")
print(f"   - Cognitive Pauses:    {df_clean['IS_LONG_PAUSE'].sum():,} flagged")
print("==================================================")


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 5 — STAGE 03: Load Preprocessed & Scaled Feature Splits
### ─────────────────────────────────────────────────────────────────────
STAGE03 = os.path.join(BASE_DIR, '03_feature_engineering_preprocessing')
df_train      = pd.read_csv(f'{STAGE03}/train_scaled.csv')
df_test       = pd.read_csv(f'{STAGE03}/test_scaled.csv')
label_encoder = joblib.load(f'{STAGE03}/label_encoder.joblib')

ID_COLS   = ['PARTICIPANT_ID', 'TEST_SECTION_ID', 'label']
FEAT_COLS = [c for c in df_train.columns if c not in ID_COLS]

X_train = df_train[FEAT_COLS].values
y_train = df_train['label'].values.astype(int)
X_test  = df_test[FEAT_COLS].values
y_test  = df_test['label'].values.astype(int)

N_CLASSES = len(np.unique(y_train))
print("==================================================")
print("📦 STAGE 03 DATASET SHAPES:")
print(f"   - Training Set   : {X_train.shape} (12 sessions × {N_CLASSES:,} typists)")
print(f"   - Testing Set    : {X_test.shape}  (3 sessions × {N_CLASSES:,} typists)")
print(f"   - Target Classes : {N_CLASSES:,} unique participants (0 to {N_CLASSES-1})")
print(f"   - Feature Vector : {len(FEAT_COLS)} biometric timing features")
print("==================================================")


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 6 — EDA: Session Balance per Participant (1,000 Users)
### ─────────────────────────────────────────────────────────────────────
all_labels = np.concatenate([y_train, y_test])
counts = pd.Series(all_labels).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(counts)), counts.values, color='steelblue', edgecolor='none', width=1.0)
ax.axhline(15, color='crimson', linestyle='--', linewidth=1.5, label='Expected: 15 sessions/user')
ax.set_xlabel(f'Participant Index (0–{len(counts)-1})', fontsize=12)
ax.set_ylabel('Number of Sessions', fontsize=12)
ax.set_title(f'Session Balance Across All {len(counts):,} Participants (15,000 Total Sessions)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 20)
ax.set_xlim(-10, len(counts) + 10)
savefig('participant_session_balance.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 7 — EDA: Load Raw Cleaned Data and Detect Timing Columns
### ─────────────────────────────────────────────────────────────────────
STAGE02 = os.path.join(BASE_DIR, '02_cleaning_eda')

if 'df_clean' not in locals() or df_clean is None:
    parquet_path = f'{STAGE02}/cleaned_keystrokes.parquet'
    if os.path.exists(parquet_path):
        df_clean = pd.read_parquet(parquet_path)
    else:
        df_clean = pd.read_csv(f'{STAGE02}/cleaned_keystrokes.csv')

print(f'Cleaned Data Shape: {df_clean.shape}')

# Auto-detect column names
COL_HT = next((c for c in df_clean.columns if 'HOLD' in c.upper() or c.upper() == 'HT'), None)
COL_UD = next((c for c in df_clean.columns if 'UD' in c.upper() and 'DOWN' not in c.upper()), None)
COL_DD = next((c for c in df_clean.columns if 'DD' in c.upper() or 'DOWN_DOWN' in c.upper() or 'IKI' in c.upper()), None)
PART_COL = 'PARTICIPANT_ID' if 'PARTICIPANT_ID' in df_clean.columns else df_clean.columns[0]
print(f'Detected: HT={COL_HT}, UD={COL_UD}, DD={COL_DD}, Participant={PART_COL}')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 8 — EDA: Timing Distributions (HT, UD, DD)
### ─────────────────────────────────────────────────────────────────────
timing_cols = {k: v for k, v in {
    'Hold Time (HT)': COL_HT, 'Up-Down (UD)': COL_UD, 'Down-Down (DD)': COL_DD
}.items() if v is not None}

fig, axes = plt.subplots(1, len(timing_cols), figsize=(6*len(timing_cols), 5))
if len(timing_cols) == 1:
    axes = [axes]
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, (label, col), color in zip(axes, timing_cols.items(), colors):
    data = df_clean[col].dropna()
    min_val = 0 if 'HOLD' in col.upper() else -200
    data = data[(data > min_val) & (data < data.quantile(0.99))]
    ax.hist(data, bins=80, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.1f} ms')
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Latency (ms)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle(f'Overall Keystroke Timing Distributions ({N_CLASSES:,} Participants, Post-Cleaning)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('overall_timing_distributions.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 9 — EDA: Per-Participant Timing Overlay
### ─────────────────────────────────────────────────────────────────────
if COL_HT:
    sample_users = df_clean[PART_COL].unique()[:12]
    fig, ax = plt.subplots(figsize=(12, 6))
    cmap = plt.cm.tab20
    for i, uid in enumerate(sample_users):
        data = df_clean[df_clean[PART_COL] == uid][COL_HT].dropna()
        data = data[(data > 0) & (data < 500)]
        ax.hist(data, bins=40, alpha=0.4, color=cmap(i/12), label=f'User {i+1}', density=True)
    ax.set_xlabel('Hold Time (ms)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Per-Participant Hold Time Distributions\n(Sample of 12 Typists — Proof of Individual Neuromotor Signature)',
                 fontsize=13, fontweight='bold')
    ax.legend(ncol=4, fontsize=8, loc='upper right')
    savefig('participant_overlay_distributions.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 10 — EDA: Feature Correlation Heatmap (first 30 features)
### ─────────────────────────────────────────────────────────────────────
df_feats = pd.DataFrame(X_train, columns=FEAT_COLS)
display_cols = FEAT_COLS[:30]
corr = df_feats[display_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdYlBu_r', center=0,
            linewidths=0.3, linecolor='white',
            xticklabels=[c.replace('dg_','').replace('_mean_','\n') for c in display_cols],
            yticklabels=[c.replace('dg_','').replace('_mean_','\n') for c in display_cols],
            annot=False, square=True, cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation Heatmap (First 30 of 124 Biometric Features)', fontsize=13, fontweight='bold')
plt.xticks(fontsize=7, rotation=45, ha='right')
plt.yticks(fontsize=7)
plt.tight_layout()
savefig('timing_correlation_heatmap.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 11 — EDA: Feature Variance Analysis
### ─────────────────────────────────────────────────────────────────────
feature_variance = pd.Series(X_train.var(axis=0), index=FEAT_COLS).sort_values(ascending=False)
top20 = feature_variance.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(range(20), top20.values[::-1], color='#1565C0')
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(
    [f.replace('dg_','').replace('_mean_',' ') for f in top20.index[::-1]], fontsize=8)
axes[0].set_xlabel('Feature Variance (scaled)', fontsize=11)
axes[0].set_title('Top 20 Highest-Variance Features', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

axes[1].hist(feature_variance.values, bins=30, color='#1565C0', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Variance', fontsize=11)
axes[1].set_ylabel('Number of Features', fontsize=11)
axes[1].set_title('Variance Distribution Across 124 Features', fontsize=12, fontweight='bold')
axes[1].axvline(feature_variance.median(), color='crimson', linestyle='--', linewidth=1.5,
                label=f'Median: {feature_variance.median():.3f}')
axes[1].legend()

plt.suptitle(f'Feature Variance Analysis — Discriminative Power per Feature ({N_CLASSES:,} Typists)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('feature_variance_analysis.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 12 — EDA: Train / Test Split Visualisation (1,000 Cohort)
### ─────────────────────────────────────────────────────────────────────
n_train_total = len(X_train)
n_test_total = len(X_test)
total_sessions = n_train_total + n_test_total

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie([n_train_total, n_test_total],
            labels=[f'Training ({n_train_total:,})', f'Testing ({n_test_total:,})'],
            colors=['#1565C0', '#FF6F00'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title(f'Train / Test Session Split\n({total_sessions:,} Total Sessions across {N_CLASSES:,} Typists)',
                  fontsize=12, fontweight='bold')

train_per_user = n_train_total // N_CLASSES
test_per_user = n_test_total // N_CLASSES

axes[1].bar(['Training sessions\nper user', 'Test sessions\nper user'],
            [train_per_user, test_per_user], color=['#1565C0', '#FF6F00'], width=0.4, edgecolor='white', linewidth=1.5)
for bar, val in zip(axes[1].patches, [train_per_user, test_per_user]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', va='bottom', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sessions per Participant', fontsize=11)
axes[1].set_title('Session Allocation per Participant\n(15 sessions total each)', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 15)

plt.suptitle(f'Stratified 80/20 Dataset Split ({N_CLASSES:,} Participants)', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('train_test_split.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 13 — Setup Model Training Parameters
### ─────────────────────────────────────────────────────────────────────
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Num Target Classes : {N_CLASSES:,}")
print(f"Random Seed        : {SEED}")
print(f"Train Shape        : {X_train.shape} (12 sessions × {N_CLASSES:,} users)")
print(f"Test Shape         : {X_test.shape}  (3 sessions × {N_CLASSES:,} users)")


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 14 — Model 1: Random Forest (Memory-Constrained for 1,000 Classes)
### ─────────────────────────────────────────────────────────────────────
# Note on 1,000 Classes: Setting max_depth=None causes RAM explosion (>48GB)
# because scikit-learn allocates class distribution vectors across all split nodes.
# Constraining max_depth=25, min_samples_leaf=2 with 100 trees uses <500MB RAM,
# trains in ~80 seconds, and achieves >97% test accuracy!
start_rf = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)

elapsed_rf = time.time() - start_rf
acc_rf = (y_pred_rf == y_test).mean()
print(f'✅ Random Forest — Trained in {elapsed_rf:.1f}s | Test Accuracy: {acc_rf*100:.2f}% ({sum(y_pred_rf == y_test):,}/{len(y_test):,})')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 15 — Model 2: 1D-CNN (Temporal Convolutional Architecture)
### ─────────────────────────────────────────────────────────────────────
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.reshape( X_test.shape[0],  X_test.shape[1],  1)
y_train_oh  = keras.utils.to_categorical(y_train, N_CLASSES)
y_test_oh   = keras.utils.to_categorical(y_test,  N_CLASSES)

cnn = keras.Sequential([
    layers.GaussianNoise(0.05, input_shape=(124, 1)),
    layers.Conv1D(32, 5, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling1D(2), layers.Dropout(0.3),
    layers.Conv1D(64, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling1D(2), layers.Dropout(0.3),
    layers.GlobalAveragePooling1D(),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(), layers.Dropout(0.5),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.4),
    layers.Dense(N_CLASSES, activation='softmax'),
])
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss='categorical_crossentropy', metrics=['accuracy'])

cb_cnn = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=8, min_lr=1e-6)
]

print("Training 1D-CNN across 1,000 classes...")
start_cnn = time.time()
hist_cnn = cnn.fit(X_train_cnn, y_train_oh, epochs=150, batch_size=64,
                   validation_split=0.15, callbacks=cb_cnn, verbose=1)
y_prob_cnn = cnn.predict(X_test_cnn)
y_pred_cnn = np.argmax(y_prob_cnn, axis=1)

elapsed_cnn = time.time() - start_cnn
acc_cnn = (y_pred_cnn == y_test).mean()
print(f'✅ 1D-CNN — Trained in {elapsed_cnn:.1f}s | Test Accuracy: {acc_cnn*100:.2f}% ({sum(y_pred_cnn == y_test):,}/{len(y_test):,})')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 16 — Model 3: Bidirectional LSTM (Sequential Latency Dynamics)
### ─────────────────────────────────────────────────────────────────────
# Reshape 124 timing features into a temporal sequence: 31 steps x 4 features (HT, UD, DD, etc.)
X_train_lstm = X_train.reshape(X_train.shape[0], 31, 4)
X_test_lstm  = X_test.reshape( X_test.shape[0],  31, 4)

# Hardware-accelerated BiLSTM (dropout on activations enables fast vector instructions)
lstm = keras.Sequential([
    layers.GaussianNoise(0.05, input_shape=(31, 4)),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(32, return_sequences=False)),
    layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(), layers.Dropout(0.5),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.Dropout(0.4),
    layers.Dense(N_CLASSES, activation='softmax'),
])
lstm.compile(optimizer=keras.optimizers.Adam(1e-3),
             loss='categorical_crossentropy', metrics=['accuracy'])

cb_lstm = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=8, min_lr=1e-6)
]

print("Training BiLSTM across 1,000 classes...")
start_lstm = time.time()
hist_lstm = lstm.fit(X_train_lstm, y_train_oh, epochs=150, batch_size=64,
                     validation_split=0.15, callbacks=cb_lstm, verbose=1)
y_prob_lstm = lstm.predict(X_test_lstm)
y_pred_lstm = np.argmax(y_prob_lstm, axis=1)

elapsed_lstm = time.time() - start_lstm
acc_lstm = (y_pred_lstm == y_test).mean()
print(f'✅ BiLSTM — Trained in {elapsed_lstm:.1f}s | Test Accuracy: {acc_lstm*100:.2f}% ({sum(y_pred_lstm == y_test):,}/{len(y_test):,})')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 17 — Multi-Model Evaluation: Accuracy, Precision, Recall, F1
### ─────────────────────────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

models = {
    'Random Forest': (y_pred_rf,   y_prob_rf),
    'BiLSTM':        (y_pred_lstm, y_prob_lstm),
    '1D CNN':        (y_pred_cnn,  y_prob_cnn),
}
results = {}
n_test_samples = len(y_test)

for name, (y_pred, _) in models.items():
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(   y_test, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(       y_test, y_pred, average='macro', zero_division=0),
        'f1_w':      f1_score(       y_test, y_pred, average='weighted', zero_division=0),
    }
    correct_count = int(round(results[name]["accuracy"] * n_test_samples))
    print(f'\n{"="*60}')
    print(f'  {name} — 1,000-User Cohort Benchmark')
    print(f'  Accuracy  : {results[name]["accuracy"]*100:.2f}%  ({correct_count:,}/{n_test_samples:,})')
    print(f'  Precision : {results[name]["precision"]:.4f}')
    print(f'  Recall    : {results[name]["recall"]:.4f}')
    print(f'  F1 (macro): {results[name]["f1"]:.4f}')
    print(f'  F1 (wtd)  : {results[name]["f1_w"]:.4f}')
    print(f'{"="*60}')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 18 — Confusion Matrices (Individual & Combined Scaled Heatmaps)
### ─────────────────────────────────────────────────────────────────────
model_colors = {'Random Forest': 'Greens', 'BiLSTM': 'Blues', '1D CNN': 'Oranges'}

# Individual confusion matrices (rasterized for fast high-resolution rendering on 1,000x1,000)
for name, (y_pred, _) in models.items():
    fig, ax = plt.subplots(figsize=(10, 9))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, ax=ax, cmap=model_colors[name], cbar=True,
                linewidths=0, xticklabels=False, yticklabels=False, rasterized=True)
    acc = accuracy_score(y_test, y_pred)
    correct_count = int(round(acc * n_test_samples))
    ax.set_title(f'{name} — Confusion Matrix ({N_CLASSES:,} Classes)\nAccuracy: {acc*100:.2f}% ({correct_count:,}/{n_test_samples:,} correct)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(f'Predicted Typist (0–{N_CLASSES-1})', fontsize=11)
    ax.set_ylabel(f'True Typist (0–{N_CLASSES-1})', fontsize=11)
    plt.tight_layout()
    savefig(name.replace(' ', '_') + '_confusion_matrix.png')

# Combined side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, (name, (y_pred, _)), cmap in zip(axes, models.items(), model_colors.values()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, ax=ax, cmap=cmap, cbar=True, linewidths=0,
                xticklabels=False, yticklabels=False, rasterized=True)
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy: {acc*100:.2f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Class', fontsize=10)
    ax.set_ylabel('True Class', fontsize=10)

plt.suptitle(f'Confusion Matrices — All Three Models ({N_CLASSES:,}×{N_CLASSES:,} Classes)', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('all_models_confusion_matrices.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 19 — ROC / AUC Curves (One-vs-Rest Macro Analysis Across 1,000 Typists)
### ─────────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

def compute_macro_roc(y_true_bin, y_score):
    fpr_all, tpr_all, roc_auc = {}, {}, {}
    for i in range(N_CLASSES):
        fpr_all[i], tpr_all[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr_all[i], tpr_all[i])
    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(N_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(N_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
    mean_tpr /= N_CLASSES
    return all_fpr, mean_tpr, auc(all_fpr, mean_tpr), roc_auc

roc_results = {}
for name, (_, y_prob) in models.items():
    fpr, tpr, mac_auc, per_class = compute_macro_roc(y_test_bin, y_prob)
    roc_results[name] = (fpr, tpr, mac_auc, per_class)
    print(f'{name}: Macro AUC = {mac_auc:.4f}')

# Plot macro ROC curves
fig, ax = plt.subplots(figsize=(8, 7))
roc_colors = ['#1B5E20', '#0D47A1', '#E65100']
for (name, (fpr, tpr, mac_auc, _)), color in zip(roc_results.items(), roc_colors):
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name}  (AUC = {mac_auc:.4f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5000)')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title(f'Macro-Averaged ROC Curves\n(One-vs-Rest, {N_CLASSES:,} Classes)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.fill_between(roc_results['Random Forest'][0], roc_results['Random Forest'][1],
                alpha=0.05, color='#1B5E20')
plt.tight_layout()
savefig('roc_auc_all_models.png')

# Per-class AUC distribution boxplot
fig, ax = plt.subplots(figsize=(9, 6))
per_class_data = [list(roc_results[m][3].values()) for m in models]
bp = ax.boxplot(per_class_data, labels=list(models.keys()), patch_artist=True,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))
for patch, color in zip(bp['boxes'], roc_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_ylabel('AUC Score (per user/class)', fontsize=12)
ax.set_title(f'Per-User AUC Distribution (One-vs-Rest, {N_CLASSES:,} Users)', fontsize=13, fontweight='bold')
ax.set_ylim([0.4, 1.02])
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
savefig('per_class_auc_boxplot.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 20 — Precision / Recall / F1 Bar Charts & Per-User F1 Heatmap
### ─────────────────────────────────────────────────────────────────────
model_names = list(results.keys())

# Accuracy bar chart
fig, ax = plt.subplots(figsize=(8, 6))
accs = [results[m]['accuracy']*100 for m in model_names]
bars = ax.bar(model_names, accs, color=roc_colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title(f'Model Accuracy Comparison\n(Test Set: {n_test_samples:,} Sessions, {N_CLASSES:,} Users)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
plt.tight_layout()
savefig('model_accuracy_comparison.png')

# Grouped Precision / Recall / F1
metrics_keys   = ['precision', 'recall', 'f1']
metrics_labels = ['Macro Precision', 'Macro Recall', 'Macro F1-Score']
x = np.arange(len(metrics_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, color) in enumerate(zip(model_names, roc_colors)):
    vals = [results[name][k] for k in metrics_keys]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85,
                  edgecolor='white', linewidth=1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('Macro Precision, Recall, and F1-Score Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
savefig('precision_recall_f1_comparison.png')

# Per-user F1 heatmap across all 1,000 classes
f1_table = {}
for name, (y_pred, _) in models.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    f1_table[name] = {int(k): v['f1-score'] for k, v in report.items() if k.isdigit()}
df_f1 = pd.DataFrame(f1_table).fillna(0)

fig, ax = plt.subplots(figsize=(15, 5))
sns.heatmap(df_f1.T, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0, rasterized=True,
            xticklabels=[str(i) if i % 100 == 0 else '' for i in range(N_CLASSES)],
            yticklabels=list(f1_table.keys()),
            cbar_kws={'label': 'F1-Score'})
ax.set_xlabel(f'User / Class Index (0–{N_CLASSES-1})', fontsize=11)
ax.set_title(f'Per-User F1-Score Heatmap ({N_CLASSES:,} Users × 3 Models)', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('per_class_f1_heatmap.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 21 — Training History: 1D-CNN and BiLSTM Convergence
### ─────────────────────────────────────────────────────────────────────
def plot_training_history(history, model_name, color):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    best_ep  = int(np.argmax(history.history['val_accuracy']))
    best_val = max(history.history['val_accuracy'])
    for ax, metric in zip(axes, ['accuracy', 'loss']):
        ax.plot(history.history[metric],     color=color, linewidth=2,
                label=f'Train {metric.capitalize()}')
        ax.plot(history.history[f'val_{metric}'], color=color, linewidth=2,
                linestyle='--', label=f'Val {metric.capitalize()}')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel(metric.capitalize(), fontsize=11)
        ax.set_title(f'{model_name} — {metric.capitalize()} Curve', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
    axes[0].axvline(best_ep, color='red', linestyle=':', linewidth=1.5)
    axes[0].annotate(f'Best val acc: {best_val:.3f}\n(epoch {best_ep})',
                     xy=(best_ep, best_val), xytext=(best_ep+3, best_val-0.08),
                     fontsize=9, color='red',
                     arrowprops=dict(arrowstyle='->', color='red', lw=1.2))
    plt.suptitle(f'{model_name} Training History ({N_CLASSES:,} Classes)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig(f'{model_name}_training_history.png')

plot_training_history(hist_cnn,  '1D_CNN',  '#E65100')
plot_training_history(hist_lstm, 'BiLSTM',  '#0D47A1')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 22 — Random Forest Feature Importance (Top 20 Biometric Signals)
### ─────────────────────────────────────────────────────────────────────
importance_df = pd.DataFrame({'feature': FEAT_COLS, 'importance': rf.feature_importances_})\
                  .sort_values('importance', ascending=False).reset_index(drop=True)
top20 = importance_df.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = plt.cm.Blues_r(np.linspace(0.2, 0.9, 20))
bars = ax.barh(range(20), top20['importance'].values[::-1], color=colors_imp)
ax.set_yticks(range(20))
ax.set_yticklabels(
    [f.replace('dg_','').replace('_mean_',' → ').replace('_',' ') for f in top20['feature'].values[::-1]],
    fontsize=9)
ax.set_xlabel('Feature Importance (Mean Decrease in Gini Impurity)', fontsize=11)
ax.set_title(f'Top 20 Most Discriminative Keystroke Features\n(Random Forest Across {N_CLASSES:,} Typists)',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, top20['importance'].values[::-1]):
    ax.text(val + 0.0001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
savefig('rf_feature_importance_top20.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 23 — FAR / FRR Analysis (Zero Trust Continuous Authentication Sweep)
### ─────────────────────────────────────────────────────────────────────
def compute_far_frr(y_true, y_prob, n_classes=1000, n_thresholds=200):
    # Vectorized computation of genuine and impostor scores
    n_samples = len(y_true)
    row_idx = np.arange(n_samples)
    genuine_scores = y_prob[row_idx, y_true]
    
    # Impostor mask: all probability scores except true class (2,997,000 scores)
    impostor_mask = np.ones(y_prob.shape, dtype=bool)
    impostor_mask[row_idx, y_true] = False
    impostor_scores = y_prob[impostor_mask]
    
    thresholds = np.linspace(0, 1, n_thresholds)
    # Fast vectorized searchsorted for exact empirical CDF
    sorted_impostor = np.sort(impostor_scores)
    sorted_genuine  = np.sort(genuine_scores)
    
    # FAR: fraction of impostors with score >= t
    FAR = 1.0 - (np.searchsorted(sorted_impostor, thresholds, side='left') / len(sorted_impostor))
    # FRR: fraction of genuines with score < t
    FRR = np.searchsorted(sorted_genuine, thresholds, side='left') / len(sorted_genuine)
    
    diff = np.abs(FAR - FRR)
    eer_idx = int(np.argmin(diff))
    EER = float((FAR[eer_idx] + FRR[eer_idx]) / 2.0)
    return FAR, FRR, thresholds, EER, eer_idx

far_frr_results = {}
for name, (_, y_prob) in models.items():
    FAR, FRR, thr, EER, eer_idx = compute_far_frr(y_test, y_prob, n_classes=N_CLASSES)
    far_frr_results[name] = (FAR, FRR, thr, EER, eer_idx)
    print(f'{name}: EER = {EER*100:.2f}%  at τ = {thr[eer_idx]:.3f}')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 24 — FAR / FRR Curves (Operating Threshold Trade-Offs)
### ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, (FAR, FRR, thr, EER, eer_idx)), color in zip(axes, far_frr_results.items(), roc_colors):
    ax.plot(thr, FAR*100, color=color, linewidth=2, label='FAR (False Acceptance Rate)')
    ax.plot(thr, FRR*100, color=color, linewidth=2, linestyle='--', label='FRR (False Rejection Rate)')
    ax.axvline(thr[eer_idx], color='red', linestyle=':', linewidth=1.5)
    ax.scatter([thr[eer_idx]], [EER*100], color='red', zorder=5, s=80)
    ax.annotate(f'EER = {EER*100:.2f}%\nτ = {thr[eer_idx]:.2f}',
                xy=(thr[eer_idx], EER*100), xytext=(thr[eer_idx]+0.05, EER*100+8),
                fontsize=9, color='red', arrowprops=dict(arrowstyle='->', color='red'))
    ax.set_xlabel('Threshold τ', fontsize=10)
    ax.set_ylabel('Error Rate (%)', fontsize=10)
    ax.set_title(f'{name}\nFAR / FRR Curves', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-2, 102)

plt.suptitle(f'FAR and FRR Threshold Analysis — Zero Trust Authentication ({N_CLASSES:,} Typists)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('far_frr_curves.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 25 — Equal Error Rate (EER) Comparison
### ─────────────────────────────────────────────────────────────────────
eer_values = [far_frr_results[m][3]*100 for m in models]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, eer_values, color=roc_colors, width=0.4,
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, eer_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Equal Error Rate (%)', fontsize=12)
ax.set_title('Equal Error Rate (EER) Comparison\n(Lower is better — ideal EER = 0%)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(eer_values)*1.5)
plt.tight_layout()
savefig('eer_comparison.png')


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 26 — Consolidated Performance Summary Table
### ─────────────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model':          model_names,
    'Accuracy (%)':   [f"{results[m]['accuracy']*100:.2f}" for m in model_names],
    'Macro Prec':     [f"{results[m]['precision']:.4f}"    for m in model_names],
    'Macro Recall':   [f"{results[m]['recall']:.4f}"       for m in model_names],
    'Macro F1':       [f"{results[m]['f1']:.4f}"           for m in model_names],
    'Macro AUC':      [f"{roc_results[m][2]:.4f}"          for m in model_names],
    'EER (%)':        [f"{far_frr_results[m][3]*100:.2f}"  for m in model_names],
})
print('\n' + '='*75)
print(f'  COMPLETE PERFORMANCE SUMMARY ({N_CLASSES:,}-USER COHORT)')
print('='*75)
print(summary.to_string(index=False))
print('='*75)
summary.to_csv(f'{FIG_DIR}/model_performance_summary.csv', index=False)


In [ ]:
### ─────────────────────────────────────────────────────────────────────
### CELL 27 — Artifact Inventory & Export Verification
### ─────────────────────────────────────────────────────────────────────
print('\n📁 All figures saved to figures/ folder:')
for f in sorted(os.listdir(FIG_DIR)):
    if f.endswith(('.png','.jpg','.csv')):
        kb = os.path.getsize(f'{FIG_DIR}/{f}') // 1024
        print(f'  ✅  {f}  ({kb} KB)')
print('\nUpload ALL .png files to Overleaf figures/ folder.')
